MinerU 轻量级

In [ ]:
import requests
import time
import os
from tqdm import tqdm
import pdfplumber

BASE_URL = "https://mineru.net/api/v1/agent"

def parse_by_file(file_path, language="ch", page_range=None, enable_table=True, is_ocr=False, enable_formula=True):
    """通过文件上传提交文档解析任务并等待结果。"""
    file_name = file_path.split("/")[-1].split("\\")[-1]

    # 1. 获取签名上传 URL
    data = {"file_name": file_name, "language": language, "enable_table": enable_table, "is_ocr": is_ocr, "enable_formula": enable_formula}
    if page_range:
        data["page_range"] = page_range

    resp = requests.post(f"{BASE_URL}/parse/file", json=data)
    result = resp.json()
    if result["code"] != 0:
        print(f"获取上传链接失败: {result['msg']}")
        return None

    task_id = result["data"]["task_id"]
    file_url = result["data"]["file_url"]
    print(f"任务已创建, task_id: {task_id}")

    # 2. PUT 上传文件到 OSS
    with open(file_path, "rb") as f:
        put_resp = requests.put(file_url, data=f)
        if put_resp.status_code not in (200, 201):
            print(f"文件上传失败, HTTP {put_resp.status_code}")
            return None
    print("文件上传成功，等待解析...")

    # 3. 轮询等待结果
    return poll_result(task_id)


def poll_result(task_id, timeout=300, interval=3):
    """轮询查询解析结果。"""
    state_labels = {
        "pending": "排队中",
        "running": "解析中",
        "waiting-file": "等待文件上传",
    }
    start = time.time()
    while time.time() - start < timeout:
        resp = requests.get(f"{BASE_URL}/parse/{task_id}")
        result = resp.json()
        state = result["data"]["state"]
        elapsed = int(time.time() - start)

        if state == "done":
            markdown_url = result["data"]["markdown_url"]
            print(f"[{elapsed}s] 解析完成, Markdown 下载链接: {markdown_url}")
            md_resp = requests.get(markdown_url)
            md_resp.encoding = 'utf-8'
            return md_resp.text

        if state == "failed":
            print(f"[{elapsed}s] 解析失败: {result['data'].get('err_msg', '未知错误')}")
            return None

        print(f"[{elapsed}s] {state_labels.get(state, state)}...")
        time.sleep(interval)

    print(f"轮询超时 ({timeout}s)，请稍后手动查询 task_id: {task_id}")
    return None

def get_total_pages(file_path):
    """使用pdfplumber获取PDF总页数"""
    try:
        with pdfplumber.open(file_path) as pdf:
            return len(pdf.pages)
    except Exception as e:
        print(f"pdfplumber获取页数失败: {e}")
        return None

def parse_long_document(file_path, pages_per_batch=15):
    """分批次解析长文档并合并结果"""
    all_content = []
    page_start = 1
    total_pages = get_total_pages(file_path)
    
    while True:
        # 计算当前批次的结束页码
        page_end = page_start + pages_per_batch - 1
        if page_end > total_pages:
            page_end = total_pages
        
        # 调用解析函数
        content = parse_by_file(
            file_path, 
            page_range=f"{page_start}-{page_end}"
        )
        
        if content:
            all_content.append(content)
            print(f"成功解析第{page_start}-{page_end}页")
        else:
            print(f"解析第{page_start}-{page_end}页失败，停止处理")
            break
        
        # 检查是否到达文档末尾
        if page_end >= get_total_pages(file_path):  # 需要实现get_total_pages函数
            break
            
        page_start = page_end + 1
    
    return "\n".join(all_content)

# 使用示例
input_folder = 'B题数据及提交说明/全部数据/正式数据/附件5：研报数据/行业研报'
output_folder = 'B题数据及提交说明/全部数据/正式数据/附件5：研报数据/行业研报-解析结果'
os.makedirs(output_folder, exist_ok=True)
pdf_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith('.pdf')
    ]
pdf_done = [f[:-3] for f in os.listdir(output_folder)
        if f.lower().endswith('.md')]

for filename in tqdm(pdf_files, desc="处理进度"):
    if filename[:-4] in pdf_done:
        print(f"跳过已处理文件: {filename}")
        continue
    
    file_path = os.path.join(input_folder, filename)
    # 获取总页数
    total_pages = get_total_pages(file_path)
    if total_pages and total_pages > 20:
        print(f"{filename} 是长文档，共 {total_pages} 页，使用分批次解析")
        content = parse_long_document(file_path, pages_per_batch=15)
    else:
        content = parse_by_file(file_path)
    if content:
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.md")
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"已保存解析结果: {output_path}")
    else:
        print(f"解析失败: {filename}")

MinerU 精准版

In [ ]:
import requests
import time
import os
import json
import zipfile
from io import BytesIO
from tqdm import tqdm
import pdfplumber
from dotenv import load_dotenv

load_dotenv()

# === 配置区 ===
api_key = os.getenv("MINERU_API_KEY", "")
BASE_URL = "https://mineru.net/api/v4"
header = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {api_key}'
}

# 1. 修改为使用精准解析的批量上传接口 (即使只传一个文件)
def parse_by_file_v4(file_path, language="ch", page_range=None, enable_table=True, is_ocr=False, enable_formula=True, output_dir=None):
    """通过精准解析API (v4) 提交文档任务"""
    file_name = os.path.basename(file_path)
    
    # --- 步骤1: 申请上传链接 (Batch API) ---
    apply_url = f"{BASE_URL}/file-urls/batch"
    
    # 构建请求体
    data = {
        "files": [
            {
                "name": file_name,
            }
        ],
        "model_version": "vlm",
        "language": language,
        "enable_table": enable_table,
        "is_ocr": is_ocr,
        "enable_formula": enable_formula
    }
    
    if page_range:
        data["page_ranges"] = page_range

    try:
        resp = requests.post(apply_url, headers=header, json=data)
        result = resp.json()
        
        if result["code"] != 0:
            print(f"申请上传链接失败: {result['msg']}")
            return None, None

        batch_id = result["data"]["batch_id"]
        file_urls = result["data"]["file_urls"]
        upload_url = file_urls[0]
        
        print(f"获取上传链接成功, Batch ID: {batch_id}")
        
        # --- 步骤2: 上传文件 ---
        with open(file_path, "rb") as f:
            put_resp = requests.put(upload_url, data=f)
            if put_resp.status_code not in (200, 201):
                print(f"文件上传失败, HTTP {put_resp.status_code}: {put_resp.text}")
                return None, None
            
        print("文件上传成功，等待解析...")

        # --- 步骤3: 轮询结果 ---
        # 关键修改：传入output_dir
        return poll_result_v4(batch_id, is_batch=True, output_dir=output_dir), batch_id
        
    except Exception as e:
        print(f"请求异常: {e}")
        return None, None

# 2. 修改轮询逻辑以处理精准版的响应结构
def poll_result_v4(identifier, is_batch=False, timeout=600, interval=5, output_dir=None):
    """轮询精准版解析结果"""
    start = time.time()
    
    while time.time() - start < timeout:
        elapsed = int(time.time() - start)
        
        if is_batch:
            url = f"{BASE_URL}/extract-results/batch/{identifier}"
        else:
            url = f"{BASE_URL}/extract/task/{identifier}"
        
        try:
            resp = requests.get(url, headers=header)
            result = resp.json()
            
            if not is_batch:
                state = result["data"]["state"]
                if state == "done":
                    zip_url = result["data"]["full_zip_url"]
                    # 关键修改：传入output_dir
                    return process_zip_result(zip_url, output_dir)
                elif state == "failed":
                    print(f"解析失败: {result['data'].get('err_msg', '未知错误')}")
                    return None
                else:
                    print(f"[{elapsed}s] {state}...")
            else:
                batch_data = result["data"]
                extract_results = batch_data.get("extract_result", [])
                
                if not extract_results:
                    print(f"[{elapsed}s] 等待中...")
                    time.sleep(interval)
                    continue
                    
                first_task = extract_results[0]
                state = first_task["state"]
                
                if state == "done":
                    zip_url = first_task["full_zip_url"]
                    # 关键修改：传入output_dir
                    return process_zip_result(zip_url, output_dir)
                elif state == "failed":
                    print(f"解析失败: {first_task.get('err_msg', '未知错误')}")
                    return None
                else:
                    state_map = {
                        "waiting-file": "等待文件上传",
                        "pending": "排队中",
                        "running": "解析中",
                        "converting": "格式转换中"
                    }
                    display_state = state_map.get(state, state)
                    print(f"[{elapsed}s] {display_state}...")
                    
        except Exception as e:
            print(f"轮询异常: {e}")
            
        time.sleep(interval)

    print(f"轮询超时 ({timeout}s)")
    return None

# 3. 新增：处理精准版返回的 Zip 包
def process_zip_result(zip_url, output_dir):
    """下载并解压Zip包到指定目录"""
    print(f"解析完成，正在下载结果包: {zip_url}")
    
    try:
        response = requests.get(zip_url)
        if response.status_code != 200:
            print("下载结果包失败")
            return None
            
        # 在内存中解压，并将文件写入output_dir
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            # 关键修改：解压所有文件到output_dir
            z.extractall(output_dir)
            print(f"文件已解压到: {output_dir}")
            
            # 返回full.md的路径
            md_path = os.path.join(output_dir, "full.md")
            if os.path.exists(md_path):
                return md_path
            else:
                print("未找到full.md")
                return None
                
    except Exception as e:
        print(f"处理结果压缩包失败: {e}")
        return None

# --- 以下是你的业务逻辑调用部分，只需微调函数名 ---

def get_total_pages(file_path):
    """使用pdfplumber获取PDF总页数"""
    try:
        with pdfplumber.open(file_path) as pdf:
            return len(pdf.pages)
    except Exception as e:
        print(f"pdfplumber获取页数失败: {e}")
        return None

# 注意：精准版单次支持 600 页，所以你的分页逻辑可以放宽，或者保留以防超过 600 页
def parse_long_document_v4(file_path, pages_per_batch=100, output_dir=None):
    all_content = []
    page_start = 1
    total_pages = get_total_pages(file_path)
    
    if not total_pages:
        return None
        
    while page_start <= total_pages:
        page_end = min(page_start + pages_per_batch - 1, total_pages)
        
        print(f"正在解析第 {page_start} - {page_end} 页...")
        
        # 调用精准版API
        md_path, _ = parse_by_file_v4(
            file_path, 
            page_range=f"{page_start}-{page_end}",
            output_dir=output_dir
        )
        
        if md_path:
            with open(md_path, "r", encoding="utf-8") as f:
                content = f.read()
            all_content.append(content)
            print(f"成功解析第 {page_start}-{page_end} 页")
        else:
            print(f"解析第 {page_start}-{page_end} 页失败")
            break
            
        page_start = page_end + 1
    
    return "\n".join(all_content)

# --- 主程序 ---
if __name__ == "__main__":
    input_folder = '测试数据/附件5：研报数据/行业研报'
    # 创建基础输出目录
    base_output_folder = '测试数据/附件5：研报数据/行业研报-解析结果-完整版'
    os.makedirs(base_output_folder, exist_ok=True)
    
    pdf_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.pdf')]
    pdf_done = [f[:-3] for f in os.listdir(base_output_folder) if f.lower().endswith('.md')]

    for filename in tqdm(pdf_files, desc="处理进度"):
        if filename[:-4] in pdf_done:
            print(f"跳过已处理文件: {filename}")
            continue
        
        file_path = os.path.join(input_folder, filename)
        # 为每个PDF创建独立的输出目录
        pdf_name_without_ext = os.path.splitext(filename)[0]
        output_dir = os.path.join(base_output_folder, pdf_name_without_ext)
        os.makedirs(output_dir, exist_ok=True)
        
        total_pages = get_total_pages(file_path)
        
        # 精准版支持600页
        if total_pages and total_pages <= 600:
            print(f"{filename} 共 {total_pages} 页，正在一次性解析...")
            md_path, _ = parse_by_file_v4(file_path, output_dir=output_dir)
        else:
            print(f"{filename} 页数过多或未知，使用分批次解析...")
            # 注意：分批次解析也需要传入output_dir
            md_path = parse_long_document_v4(file_path, output_dir=output_dir)
            
        if md_path:
            # 将full.md复制到主输出目录
            md_dest = os.path.join(base_output_folder, f"{pdf_name_without_ext}.md")
            with open(md_path, "r", encoding="utf-8") as f:
                content = f.read()
            with open(md_dest, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"已保存解析结果: {md_dest}")
        else:
            print(f"解析失败: {filename}")

摘要插入

In [ ]:
import os
import re
import base64
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import dashscope
from dashscope import MultiModalConversation, Generation
from dotenv import load_dotenv

load_dotenv()

# 1. 设置你的 DashScope API Key
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY", "")
dashscope.api_key = DASHSCOPE_API_KEY

# 可选：设置全局 API 调用间隔（秒），避免瞬时并发过高
API_CALL_DELAY = 0.2  # 每次调用前等待，设为 0 则无延迟


def generate_caption_for_image(image_path):
    """调用 Qwen-VL 生成图片摘要"""
    time.sleep(API_CALL_DELAY)  # 限流控制
    try:
        path = Path(image_path)
        with open(path, "rb") as f:
            b64_data = base64.b64encode(f.read()).decode("utf-8")
        messages = [
            {
                'role': 'user',
                'content': [
                    {'image': f'data:image/jpeg;base64,{b64_data}'},
                    {'text': '请用中文简短总结这张图片或图表的核心内容，不超过200字。'}
                ]
            }
        ]
        response = MultiModalConversation.call(model='qwen-vl-max', messages=messages)
        if response.status_code == 200:
            caption = response.output.choices[0].message.content[0]['text'].strip()
            return f"**图表说明：** {caption}"
        else:
            print(f"API Error: {response.code}, {response.message}")
            return "**图表说明：** (AI生成失败)"
    except Exception as e:
        print(f"Error: {e}")
        return "**图表说明：** (处理出错)"


def generate_caption_for_table(table_md_text):
    """针对表格文本生成摘要（不需要图片，直接文本分析）"""
    time.sleep(API_CALL_DELAY)  # 限流控制
    try:
        prompt_text = f'下面是一段 HTML 格式的表格代码：\n{table_md_text}\n\n请用中文简短总结这张表格展示的核心数据或趋势，不超过400字。'
        response = Generation.call(model='qwen3-max', prompt=prompt_text)
        if response.status_code == 200:
            caption = response.output.choices[0].message.content
            return f"**表格说明：** {caption}"
        else:
            print(f"   [错误] API 调用失败: Code={response.status_code}, Message={response.message}")
            return "**表格说明：** (AI生成失败)"
    except Exception as e:
        print(f"Error: {e}")
        return "**表格说明：** (处理出错)"


def process_md_file(md_path, output_path):
    """处理单个 MD 文件，为其中的图片和表格添加 AI 说明（线程安全）"""
    with open(md_path, 'r', encoding='utf-8') as f:
        content = f.read()

    def replace_image(match):
        img_full = match.group(0)
        img_path = match.group(1).strip()
        name, _ = os.path.splitext(md_path)
        full_img_path = os.path.join(name, img_path)
        if os.path.exists(full_img_path):
            print(f"正在分析图片: {full_img_path}")
            ai_caption = generate_caption_for_image(full_img_path)
            return f"{img_full}\n\n{ai_caption}"
        else:
            print(f"图片未找到: {full_img_path}")
            return img_full

    def replace_html_table(match):
        table_html = match.group(0)
        print("正在分析表格...")
        ai_caption = generate_caption_for_table(table_html)
        return f"{table_html}\n\n{ai_caption}"

    # 备份原文件（在原目录下）
    original_path = md_path.replace('.md', '_original.md')
    os.rename(md_path, original_path)

    # 处理图片和表格
    content_new = re.sub(r'!\[.*?\]\((.*?)\)', replace_image, content, flags=re.DOTALL)
    content_new = re.sub(r'<table>.*?</table>', replace_html_table, content_new, flags=re.DOTALL)

    # 写入新文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content_new)

    print(f"处理完成: {output_path}")


def main(target_folder, output_folder, max_workers=5):
    """并行处理文件夹下所有 .md 文件"""
    os.makedirs(output_folder, exist_ok=True)

    # 收集需要处理的文件
    tasks = []
    for root, dirs, files in os.walk(target_folder):
        # 只处理指定文件夹下的文件，不进入子文件夹
        if root == target_folder:
            for file in files:
                if file.endswith(".md") and not file.endswith("_original.md"):
                    md_path = os.path.join(root, file)
                    out_path = os.path.join(output_folder, file)
                    if os.path.exists(out_path):
                        print(f"跳过已存在的输出文件: {out_path}")
                        continue
                    tasks.append((md_path, out_path))

    if not tasks:
        print("没有需要处理的文件。")
        return

    print(f"共 {len(tasks)} 个文件待处理，使用 {max_workers} 个线程并发。")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(process_md_file, md_path, out_path): md_path for md_path, out_path in tasks}
        for future in as_completed(future_to_file):
            md_path = future_to_file[future]
            try:
                future.result()
            except Exception as e:
                print(f"处理文件 {md_path} 时发生异常: {e}")


if __name__ == "__main__":
    # 指定输入和输出文件夹
    target_folder = "测试数据/附件5：研报数据/行业研报-解析结果-完整版"
    output_folder = "测试数据/附件5：研报数据/行业研报-解析结果-完整版-2.0"

    # 可根据 API 限流情况调整 max_workers（建议 3~5）
    main(target_folder, output_folder, max_workers=5)

In [ ]:
# 查询target_folder里所有_original.md文件，将_original.md前的字段提取出来，将这些字段对应的.md文件删除，并将_original.md文件重命名为.md
import os
target_folder = "测试数据/附件5：研报数据/个股研报-解析结果-完整版"
for root, dirs, files in os.walk(target_folder):
    if root == target_folder:
        for file in files:
            if file.endswith("_original.md") and "+" in file:
                original_path = os.path.join(root, file)
                new_name = file.replace("_original.md", ".md")
                new_path = os.path.join(root, new_name)
                
                # 重命名 _original.md 为 .md
                os.rename(original_path, new_path)
                print(f"已重命名: {original_path} -> {new_path}")